<a href="https://colab.research.google.com/github/rafaeldrrmachado/GBD_atp_database/blob/main/TMCD_tweetsentimentanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Iscte/TM/data

/content/drive/MyDrive/Iscte/TM/data


In [ ]:
print("Estou na diretoria", os.getcwd())

Estou na diretoria /content/drive/MyDrive/Iscte/TM/data


In [ ]:
import pandas as pd

In [ ]:
df_train = pd.read_csv("/content/drive/MyDrive/Iscte/TM/data/Tweets_EN_sentiment_train.csv")

In [ ]:
df_test = pd.read_csv("/content/drive/MyDrive/Iscte/TM/data/Tweets_EN_sentiment_test.csv")

In [ ]:
len(df_test)

2122

In [ ]:
empty_text_count = (df_test['text'].isnull()).sum()
print(f"Tweets vazios: {empty_text_count}")

Tweets vazios: 22


In [ ]:
df_test = df_test.dropna()
print(f"Número de tweets após remover nulls: {len(df_test)}")

Número de tweets após remover nulls: 2100


In [ ]:
df_test['class'].value_counts()

,count
class,
pos,1770
neg,330


In [ ]:
len(df_train)

47799

In [ ]:
empty_text_count = (df_train['text'].isnull()).sum()
print(f"Tweets vazios: {empty_text_count}")

Tweets vazios: 272


In [ ]:
df_train = df_train.dropna()
print(f"Número de tweets após remover nulls: {len(df_train)}")

Número de tweets após remover nulls: 47527


In [ ]:
df_train['class'].value_counts()

,count
class,
pos,39417
neg,8110


In [ ]:
# Para utilizar df teste em modelos já criados
df = df_test

In [ ]:
df.head(10)

,tweet,text,class
0,1229979438,lmao i love it.,pos
1,1228603001,"Never been to Australia, but I'll keep that in...",pos
2,1228875414,Tired as shit...but what else is new...and don...,neg
3,1231134051,Levi's!,pos
4,1229262160,"no sweetie, its not love, probably just heart ...",pos
5,1230826337,"I am eating chocolate chips, pecans and peanut...",pos
6,1230670694,"Yes, that frank lloyd wright house, a man has ...",pos
7,1229202346,"Yeah, pretty much...",pos
8,1230058321,"ben gaar, ook nog zakelijk diner vanavond. eve...",pos
9,1229379429,"My photoshop skills lag, but I can assure you ...",pos


# Pré-processamento

In [ ]:
import re

In [ ]:
def preprocess_text_soft(text):
    #text = text.lower() # Converter para minúsculas
    #text = re.sub(r'USER', '', text) # Remover o USER (substituto de usernames para anonimizar os tweets)
    #text = re.sub(r'#', '', text) # Remover o símbolo de hashtag
    #text = re.sub(r'RT', '', text) # Remover 'RT' (retweets)
    #text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remover URLs
    #text = re.sub(r'\b\S+\.(?:com|org|net|edu|gov)\b\S*', '', text) # Remover mais URLs
    #text = re.sub(r'[^a-z\s]', '', text) # Remover caracteres especiais e números, mantendo apenas letras e espaços
    text = re.sub(r'\s+', ' ', text).strip() # Remover espaços extras
    return text

def preprocess_text_hard(text):
    text = text.lower() # Converter para minúsculas
    text = re.sub(r'USER', '', text) # Remover o USER (substituto de usernames para anonimizar os tweets)
    text = re.sub(r'#', '', text) # Remover o símbolo de hashtag
    text = re.sub(r'RT', '', text) # Remover 'RT' (retweets)
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remover URLs
    text = re.sub(r'\b\S+\.(?:com|org|net|edu|gov)\b\S*', '', text) # Remover mais URLs
    text = re.sub(r'[^a-z\s]', '', text) # Remover caracteres especiais e números, mantendo apenas letras e espaços
    text = re.sub(r'\s+', ' ', text).strip() # Remover espaços extras
    return text

# Ensure the 'text' column is filled with empty strings for NaN values and then converted to string type
df['processed_text_soft'] = df['text'].fillna('').astype(str).apply(preprocess_text_soft)
df['processed_text_hard'] = df['text'].fillna('').astype(str).apply(preprocess_text_hard)

## Modelos baseados em léxicos e regras

### Aplicar TextBlob

In [ ]:
from textblob import TextBlob

In [ ]:
def textblob_sentiment_analysis(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity

df['polarity'] = df['processed_text_soft'].apply(textblob_sentiment_analysis)

In [ ]:
def textblob_label_sentiment(value):
    if value >= 0:
        return 'pos'
    else:
        return 'neg'

df['sentiment'] = df['polarity'].apply(textblob_label_sentiment)

display(df[['processed_text_soft', 'sentiment','polarity','class']].head(10))

,processed_text_soft,sentiment,polarity,class
0,lmao i love it.,pos,0.550000,pos
1,"Never been to Australia, but I'll keep that in...",pos,0.000000,pos
2,Tired as shit...but what else is new...and don...,neg,-0.450000,neg
3,Levi's!,pos,0.000000,pos
4,"no sweetie, its not love, probably just heart ...",pos,0.050000,pos
5,"I am eating chocolate chips, pecans and peanut...",neg,-0.200000,pos
6,"Yes, that frank lloyd wright house, a man has ...",pos,0.000000,pos
7,"Yeah, pretty much...",pos,0.225000,pos
8,"ben gaar, ook nog zakelijk diner vanavond. eve...",pos,0.000000,pos
9,"My photoshop skills lag, but I can assure you ...",pos,0.433333,pos


In [ ]:
# Accuracy
correct_predictions = (df['class'] == df['sentiment']).sum()
total_predictions = len(df)
accuracy = correct_predictions / total_predictions

print(f"Número de previsões corretas: {correct_predictions}")
print(f"Número total de previsões: {total_predictions}")
print(f"Taxa de corretos (Accuracy): {accuracy:.2f}")

Número de previsões corretas: 1695
Número total de previsões: 2100
Taxa de corretos (Accuracy): 0.81


In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

y_true = df['class']
y_pred = df['sentiment']

# Definir a classe positiva como 'pos'
precision = precision_score(y_true, y_pred, pos_label='pos')
recall = recall_score(y_true, y_pred, pos_label='pos')
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label='pos')

print(f"--- Métricas TextBlob (Classe Positiva: 'pos') ---")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

--- Métricas TextBlob (Classe Positiva: 'pos') ---
Accuracy: 0.81
Precision: 0.88
Recall: 0.89
F1-score: 0.89


### Aplicar VADER

In [ ]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
sid = SentimentIntensityAnalyzer()

In [ ]:
def vader_sentiment_analysis(text):
    # O VADER devolve um dicionário com 'neg', 'neu', 'pos' e 'compound'
    # Usamos o 'compound' para decidir a classe final
    score = sid.polarity_scores(str(text))['compound']

    # Definir o limiar (threshold)
    if score >= 0:
        return 'pos'
    else:
        return 'neg'

# Aplicar ao DataFrame usando a limpeza "soft"
df['vader_sentiment'] = df['processed_text_soft'].apply(vader_sentiment_analysis)

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

y_true = df['class']
y_pred = df['vader_sentiment']

# Definir a classe positiva como 'pos'
precision = precision_score(y_true, y_pred, pos_label='pos')
recall = recall_score(y_true, y_pred, pos_label='pos')
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label='pos')

print(f"--- Métricas VADER (Classe Positiva: 'pos') ---")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

--- Métricas VADER (Classe Positiva: 'pos') ---
Accuracy: 0.83
Precision: 0.90
Recall: 0.89
F1-score: 0.90


## Modelos baseados em transformadores

### Distillbert

In [ ]:
#%pip install -q transformers
from transformers import pipeline

In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis", "distilbert-base-uncased-finetuned-sst-2-english")
sentiments = sentiment_pipeline(df['text'].tolist())
# Display the first 10 sentiments for review
print(sentiments[:10])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998750686645508}, {'label': 'POSITIVE', 'score': 0.9878376722335815}, {'label': 'NEGATIVE', 'score': 0.9989619255065918}, {'label': 'POSITIVE', 'score': 0.9997182488441467}, {'label': 'NEGATIVE', 'score': 0.985824465751648}, {'label': 'NEGATIVE', 'score': 0.9966475367546082}, {'label': 'POSITIVE', 'score': 0.937800943851471}, {'label': 'POSITIVE', 'score': 0.9996248483657837}, {'label': 'NEGATIVE', 'score': 0.9801746606826782}, {'label': 'NEGATIVE', 'score': 0.9974948167800903}]


In [ ]:
df['distilbert_label'] = [s['label'] for s in sentiments]
df['distilbert_score'] = [s['score'] for s in sentiments]

display(df[['text', 'class', 'distilbert_label', 'distilbert_score']].head(10))

,text,class,distilbert_label,distilbert_score
0,lmao i love it.,pos,POSITIVE,0.999875
1,"Never been to Australia, but I'll keep that in...",pos,POSITIVE,0.987838
2,Tired as shit...but what else is new...and don...,neg,NEGATIVE,0.998962
3,Levi's!,pos,POSITIVE,0.999718
4,"no sweetie, its not love, probably just heart ...",pos,NEGATIVE,0.985824
5,"I am eating chocolate chips, pecans and peanut...",pos,NEGATIVE,0.996648
6,"Yes, that frank lloyd wright house, a man has ...",pos,POSITIVE,0.937801
7,"Yeah, pretty much...",pos,POSITIVE,0.999625
8,"ben gaar, ook nog zakelijk diner vanavond. eve...",pos,NEGATIVE,0.980175
9,"My photoshop skills lag, but I can assure you ...",pos,NEGATIVE,0.997495


In [ ]:
df['distilbert_label'] = df['distilbert_label'].replace({'POSITIVE': 'pos', 'NEGATIVE': 'neg'})
display(df[['text', 'class', 'distilbert_label', 'distilbert_score']].head(10))

,text,class,distilbert_label,distilbert_score
0,lmao i love it.,pos,pos,0.999875
1,"Never been to Australia, but I'll keep that in...",pos,pos,0.987838
2,Tired as shit...but what else is new...and don...,neg,neg,0.998962
3,Levi's!,pos,pos,0.999718
4,"no sweetie, its not love, probably just heart ...",pos,neg,0.985824
5,"I am eating chocolate chips, pecans and peanut...",pos,neg,0.996648
6,"Yes, that frank lloyd wright house, a man has ...",pos,pos,0.937801
7,"Yeah, pretty much...",pos,pos,0.999625
8,"ben gaar, ook nog zakelijk diner vanavond. eve...",pos,neg,0.980175
9,"My photoshop skills lag, but I can assure you ...",pos,neg,0.997495


In [ ]:
# Accuracy
correct_predictions = (df['class'] == df['distilbert_label']).sum()
total_predictions = len(df)
accuracy = correct_predictions / total_predictions

print(f"Número de previsões corretas: {correct_predictions}")
print(f"Número total de previsões: {total_predictions}")
print(f"Taxa de corretos (Accuracy): {accuracy:.2f}")

Número de previsões corretas: 1318
Número total de previsões: 2100
Taxa de corretos (Accuracy): 0.63


In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

y_true = df['class']
y_pred = df['distilbert_label']

# Definir a classe positiva como 'pos'
precision = precision_score(y_true, y_pred, pos_label='pos')
recall = recall_score(y_true, y_pred, pos_label='pos')
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label='pos')

print(f"--- Métricas Distilbert (Classe Positiva: 'pos') ---")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

--- Métricas Distilbert (Classe Positiva: 'pos') ---
Accuracy: 0.63
Precision: 0.95
Recall: 0.59
F1-score: 0.73


### Roberta

In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis", "cardiffnlp/twitter-roberta-base-sentiment-latest")
sentiments = sentiment_pipeline(df['text'].tolist())
# Display the first 10 sentiments for review
print(sentiments[:10])

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'positive', 'score': 0.9715501666069031}, {'label': 'neutral', 'score': 0.6111265420913696}, {'label': 'negative', 'score': 0.9385328888893127}, {'label': 'neutral', 'score': 0.6474193930625916}, {'label': 'negative', 'score': 0.8140029311180115}, {'label': 'negative', 'score': 0.5053600072860718}, {'label': 'neutral', 'score': 0.6052374243736267}, {'label': 'neutral', 'score': 0.7773765325546265}, {'label': 'neutral', 'score': 0.8393235206604004}, {'label': 'negative', 'score': 0.51039057970047}]


In [ ]:
df['roberta_label'] = [s['label'] for s in sentiments]
df['roberta_score'] = [s['score'] for s in sentiments]

display(df[['text', 'class', 'roberta_label', 'roberta_score']].head(10))

,text,class,roberta_label,roberta_score
0,lmao i love it.,pos,positive,0.971550
1,"Never been to Australia, but I'll keep that in...",pos,neutral,0.611127
2,Tired as shit...but what else is new...and don...,neg,negative,0.938533
3,Levi's!,pos,neutral,0.647419
4,"no sweetie, its not love, probably just heart ...",pos,negative,0.814003
5,"I am eating chocolate chips, pecans and peanut...",pos,negative,0.505360
6,"Yes, that frank lloyd wright house, a man has ...",pos,neutral,0.605237
7,"Yeah, pretty much...",pos,neutral,0.777377
8,"ben gaar, ook nog zakelijk diner vanavond. eve...",pos,neutral,0.839324
9,"My photoshop skills lag, but I can assure you ...",pos,negative,0.510391


In [ ]:
df['roberta_label'] = df['roberta_label'].replace({'positive': 'pos', 'negative': 'neg', 'neutral': 'pos'})
display(df[['text', 'class', 'roberta_label', 'roberta_score']].head(10))

,text,class,roberta_label,roberta_score
0,lmao i love it.,pos,pos,0.971550
1,"Never been to Australia, but I'll keep that in...",pos,pos,0.611127
2,Tired as shit...but what else is new...and don...,neg,neg,0.938533
3,Levi's!,pos,pos,0.647419
4,"no sweetie, its not love, probably just heart ...",pos,neg,0.814003
5,"I am eating chocolate chips, pecans and peanut...",pos,neg,0.505360
6,"Yes, that frank lloyd wright house, a man has ...",pos,pos,0.605237
7,"Yeah, pretty much...",pos,pos,0.777377
8,"ben gaar, ook nog zakelijk diner vanavond. eve...",pos,pos,0.839324
9,"My photoshop skills lag, but I can assure you ...",pos,neg,0.510391


In [ ]:
# Accuracy
correct_predictions = (df['class'] == df['roberta_label']).sum()
total_predictions = len(df)
accuracy = correct_predictions / total_predictions

print(f"Número de previsões corretas: {correct_predictions}")
print(f"Número total de previsões: {total_predictions}")
print(f"Taxa de corretos (Accuracy): {accuracy:.2f}")

Número de previsões corretas: 1781
Número total de previsões: 2100
Taxa de corretos (Accuracy): 0.85


In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

y_true = df['class']
y_pred = df['roberta_label']

# Definir a classe positiva como 'pos'
precision = precision_score(y_true, y_pred, pos_label='pos')
recall = recall_score(y_true, y_pred, pos_label='pos')
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label='pos')

print(f"--- Métricas Distilbert (Classe Positiva: 'pos') ---")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

--- Métricas Distilbert (Classe Positiva: 'pos') ---
Accuracy: 0.85
Precision: 0.93
Recall: 0.89
F1-score: 0.91
